In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df_ab = pd.read_csv("../data/raw/retention_ab_test.csv")
print(df_ab.shape)
df_ab.head()

(3163, 15)


,customer_id,operator,circle,circle_type,plan_type,tenure_months,num_complaints_6m,network_rating,days_since_last_recharge,is_active_vlr,risk_score,group,voucher_redeemed,churned_30d,arpu_change_30d_inr
0,CUST100001,BSNL,Karnataka,Urban,Postpaid,31,0,3,11,0,0.33,Control,0,0,20.62
1,CUST100011,BSNL,West Bengal,Urban,Prepaid,16,0,4,4,0,0.36,Treatment,1,0,8.91
2,CUST100012,Vi,Bihar & Jharkhand,Rural,Prepaid,26,2,5,2,1,0.19,Treatment,0,0,1.75
3,CUST100017,Airtel,Maharashtra,Urban,Prepaid,33,0,2,27,1,0.13,Control,0,0,19.87
4,CUST100020,Airtel,Mumbai,Metro,Prepaid,8,2,3,10,1,0.15,Control,0,0,-1.22


In [2]:
print(df_ab["group"].value_counts())
print(df_ab["group"].value_counts(normalize=True))

group
Treatment    1588
Control      1575
Name: count, dtype: int64
group
Treatment    0.502055
Control      0.497945
Name: proportion, dtype: float64


In [3]:
covariates = ["tenure_months", "num_complaints_6m", "network_rating",
               "days_since_last_recharge", "risk_score", "is_active_vlr"]

comparison = df_ab.groupby("group")[covariates].mean()
print(comparison)

           tenure_months  num_complaints_6m  network_rating  \
group                                                         
Control        17.977143           0.877460        3.377143   
Treatment      17.066121           0.853904        3.381612   

           days_since_last_recharge  risk_score  is_active_vlr  
group                                                           
Control                   28.747937    0.215257       0.808889  
Treatment                 26.652393    0.217097       0.780856  


In [4]:
for col in covariates:
    treat = df_ab[df_ab["group"] == "Treatment"][col]
    control = df_ab[df_ab["group"] == "Control"][col]
    t_stat, p_val = stats.ttest_ind(treat, control)
    print(f"{col}: Treatment mean={treat.mean():.3f}, Control mean={control.mean():.3f}, p-value={p_val:.3f}")

tenure_months: Treatment mean=17.066, Control mean=17.977, p-value=0.152
num_complaints_6m: Treatment mean=0.854, Control mean=0.877, p-value=0.485
network_rating: Treatment mean=3.382, Control mean=3.377, p-value=0.907
days_since_last_recharge: Treatment mean=26.652, Control mean=28.748, p-value=0.023
risk_score: Treatment mean=0.217, Control mean=0.215, p-value=0.564
is_active_vlr: Treatment mean=0.781, Control mean=0.809, p-value=0.051


In [5]:
summary = df_ab.groupby("group").agg(
    n=("customer_id", "count"),
    churn_rate=("churned_30d", "mean"),
    voucher_redemption=("voucher_redeemed", "mean")
)
print(summary)

control_rate = summary.loc["Control", "churn_rate"]
treatment_rate = summary.loc["Treatment", "churn_rate"]
print(f"\nAbsolute difference (Control - Treatment): {control_rate - treatment_rate:.4f}")
print(f"Relative reduction: {(control_rate - treatment_rate) / control_rate:.2%}")

              n  churn_rate  voucher_redemption
group                                          
Control    1575    0.071111            0.000000
Treatment  1588    0.039043            0.608942

Absolute difference (Control - Treatment): 0.0321
Relative reduction: 45.10%


In [9]:
from statsmodels.stats.proportion import proportions_ztest

n_treat = summary.loc["Treatment", "n"]
n_control = summary.loc["Control", "n"]
churn_treat = (df_ab[df_ab["group"]=="Treatment"]["churned_30d"]).sum()
churn_control = (df_ab[df_ab["group"]=="Control"]["churned_30d"]).sum()

counts = np.array([churn_treat, churn_control])
nobs = np.array([n_treat, n_control])

z_stat, p_value = proportions_ztest(counts, nobs, alternative="two-sided")
print(f"Churned - Treatment: {churn_treat}/{n_treat}, Control: {churn_control}/{n_control}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

Churned - Treatment: 62/1588, Control: 112/1575
Z-statistic: -3.9551
P-value: 0.0001


In [10]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_low, ci_upp = confint_proportions_2indep(
    churn_control, n_control, churn_treat, n_treat, method="wald"
)
print(f"95% CI for (Control rate - Treatment rate): ({ci_low:.4f}, {ci_upp:.4f})")

95% CI for (Control rate - Treatment rate): (0.0162, 0.0479)


In [11]:
arpu_summary = df_ab.groupby("group")["arpu_change_30d_inr"].agg(["mean", "std", "count"])
print(arpu_summary)

treat_arpu = df_ab[df_ab["group"]=="Treatment"]["arpu_change_30d_inr"]
control_arpu = df_ab[df_ab["group"]=="Control"]["arpu_change_30d_inr"]

t_stat, p_val = stats.ttest_ind(treat_arpu, control_arpu)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.6f}")

                mean        std  count
group                                 
Control     0.738787  15.038921   1575
Treatment  24.161688  18.765879   1588

T-statistic: 38.7160
P-value: 0.000000


In [12]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

p1 = summary.loc["Control", "churn_rate"]
p2 = summary.loc["Treatment", "churn_rate"]
effect_size = proportion_effectsize(p1, p2)  # Cohen's h

analysis = NormalIndPower()
achieved_power = analysis.power(effect_size=effect_size, nobs1=n_control, alpha=0.05, ratio=n_treat/n_control)
print(f"Effect size (Cohen's h): {effect_size:.4f}")
print(f"Achieved power: {achieved_power:.4f}")

# What sample size WOULD have been needed for 80% power, given this effect size?
required_n = analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.80, ratio=1.0)
print(f"Required n per group for 80% power: {required_n:.0f}")

Effect size (Cohen's h): 0.1421
Achieved power: 0.9791
Required n per group for 80% power: 778


In [13]:
df_ab["complaint_segment"] = np.where(df_ab["num_complaints_6m"] >= 2, "High Complaints (2+)", "Low Complaints (0-1)")

seg_summary = df_ab.groupby(["complaint_segment", "group"])["churned_30d"].agg(["mean", "count"])
print(seg_summary)

                                    mean  count
complaint_segment    group                     
High Complaints (2+) Control    0.054422    441
                     Treatment  0.019802    404
Low Complaints (0-1) Control    0.077601   1134
                     Treatment  0.045608   1184


In [14]:
for seg in df_ab["complaint_segment"].unique():
    sub = df_ab[df_ab["complaint_segment"] == seg]
    treat = sub[sub["group"]=="Treatment"]
    control = sub[sub["group"]=="Control"]
    
    counts = np.array([treat["churned_30d"].sum(), control["churned_30d"].sum()])
    nobs = np.array([len(treat), len(control)])
    
    z_stat, p_val = proportions_ztest(counts, nobs, alternative="two-sided")
    diff = control["churned_30d"].mean() - treat["churned_30d"].mean()
    print(f"{seg}: n_treat={len(treat)}, n_control={len(control)}, "
          f"diff={diff:.4f}, p-value={p_val:.4f}")

Low Complaints (0-1): n_treat=1184, n_control=1134, diff=0.0320, p-value=0.0013
High Complaints (2+): n_treat=404, n_control=441, diff=0.0346, p-value=0.0084


In [15]:
for vlr_status, label in [(1, "Active (VLR=1)"), (0, "Inactive (VLR=0)")]:
    sub = df_ab[df_ab["is_active_vlr"] == vlr_status]
    treat = sub[sub["group"]=="Treatment"]
    control = sub[sub["group"]=="Control"]
    
    counts = np.array([treat["churned_30d"].sum(), control["churned_30d"].sum()])
    nobs = np.array([len(treat), len(control)])
    
    z_stat, p_val = proportions_ztest(counts, nobs, alternative="two-sided")
    diff = control["churned_30d"].mean() - treat["churned_30d"].mean()
    print(f"{label}: n_treat={len(treat)}, n_control={len(control)}, "
          f"diff={diff:.4f}, p-value={p_val:.4f}")

Active (VLR=1): n_treat=1240, n_control=1274, diff=0.0427, p-value=0.0000
Inactive (VLR=0): n_treat=348, n_control=301, diff=0.0013, p-value=0.9576


In [16]:
# Focus on active (VLR=1) high-risk customers - where the effect is real
active_segment = df_ab[df_ab["is_active_vlr"] == 1]
active_eligible_total = len(active_segment)  # in the full eligible (high-risk) population

# Churn reduction for this segment
churn_reduction_active = 0.0427  # from our z-test above

# Assume average plan amount as proxy for monthly revenue per customer
avg_plan_amount = df_ab.merge(
    pd.read_csv("../data/raw/indian_telecom_churn.csv")[["customer_id", "plan_amount_inr"]],
    on="customer_id"
)["plan_amount_inr"].mean()

print(f"Active high-risk customers in eligible pool: {active_eligible_total}")
print(f"Average plan amount: ₹{avg_plan_amount:.2f}/month")

# Customers retained = active_eligible_total * churn_reduction_active
customers_retained = active_eligible_total * churn_reduction_active
print(f"Estimated additional customers retained per 30-day cycle: {customers_retained:.1f}")

# Assume retained customer generates avg_plan_amount per month for, say, 6 more months (conservative LTV horizon)
retention_horizon_months = 6
monthly_revenue_saved = customers_retained * avg_plan_amount
total_value_saved = monthly_revenue_saved * retention_horizon_months

print(f"Estimated monthly revenue retained: ₹{monthly_revenue_saved:,.2f}")
print(f"Estimated value over {retention_horizon_months}-month horizon: ₹{total_value_saved:,.2f}")

# Cost side: voucher cost (10% of plan amount, redeemed at ~61% rate per our data)
voucher_redemption_rate = df_ab[df_ab['group']=='Treatment']['voucher_redeemed'].mean()
voucher_cost_per_redemption = avg_plan_amount * 0.10
total_voucher_cost = active_eligible_total * voucher_redemption_rate * voucher_cost_per_redemption

print(f"\nVoucher redemption rate: {voucher_redemption_rate:.2%}")
print(f"Estimated total voucher cost (active segment): ₹{total_voucher_cost:,.2f}")
print(f"Net estimated benefit (6-month horizon): ₹{total_value_saved - total_voucher_cost:,.2f}")

Active high-risk customers in eligible pool: 2514
Average plan amount: ₹329.84/month
Estimated additional customers retained per 30-day cycle: 107.3
Estimated monthly revenue retained: ₹35,408.04
Estimated value over 6-month horizon: ₹212,448.25

Voucher redemption rate: 60.89%
Estimated total voucher cost (active segment): ₹50,495.19
Net estimated benefit (6-month horizon): ₹161,953.06


## Phase 4 Summary: A/B Test Analysis

1. **Randomization check**: 5 of 6 pre-experiment covariates balanced (p>0.05); 
   days_since_last_recharge (p=0.023) and is_active_vlr (p=0.051) were borderline, 
   but pointed in opposite directions - consistent with normal multiple-comparison 
   noise rather than systematic imbalance.

2. **Primary outcome (30-day churn)**: Treatment 3.90% vs Control 7.11% - a 3.21pp 
   (45.1% relative) reduction. Two-proportion z-test: p=0.0001. 95% CI for the 
   difference: (1.62pp, 4.79pp). H0 rejected.

3. **Secondary outcome (ARPU change)**: Treatment +₹24.16 vs Control +₹0.74, 
   p<0.000001 - though noted this may be partly mechanical (driven by the lower 
   churn rate itself) and flagged for further investigation.

4. **Power analysis**: Cohen's h=0.142 (smaller than "small" by convention), but 
   achieved power=0.979 given our sample sizes (~1580/group vs ~778/group required 
   for 80% power) - the experiment was well-powered; a null result would have been 
   strong evidence of no effect, not just "inconclusive."

5. **Segment analysis (heterogeneous treatment effects)**:
   - High complaints (2+): 3.46pp reduction, p=0.0084
   - Low complaints (0-1): 3.20pp reduction, p=0.0013
   - **VLR-active customers: 4.27pp reduction, p<0.0001**
   - **VLR-inactive customers: 0.13pp reduction, p=0.96 (no effect)**

6. **Recommendation**: Roll out the retention voucher to VLR-active high-risk 
   customers (~79% of the eligible population) - estimated net benefit of 
   ₹1,61,953 over a 6-month horizon for this segment (2,514 customers), after 
   accounting for voucher redemption costs. For VLR-inactive customers, this 
   intervention shows no effect; a different retention strategy (e.g. re-engagement 
   campaign) should be tested for that segment.

**Next**: Phase 5 - Business translation & dashboard (Streamlit), and Phase 6 - 
packaging the full project for recruiters (README, write-up, video walkthrough).